# TEA-GNN on Wikidata12K Cross-Lingual Temporal Entity Alignment

**TEA-GNN:** Xu, Su, Lehmann, EMNLP 2021 — *Time-aware Graph Neural Network for Entity Alignment between Temporal Knowledge Graphs*.

This notebook is a **PyTorch port** of TEA-GNN for the **Wikidata12K cross-lingual** dataset (`en_fr`, `en_fa`, `en_zh`, `en_ja`).

TEA-GNN extends **RREA** with a **time-aware attention** branch (relation + timestamp orthogonal transforms). The structure mirrors the official TF code (`soledad921/TEA-GNN`) and the RREA port used in this project.

### Dataset files (per language pair)
- `quadruples_en.tsv` / `quadruples_<lang>.tsv` — `head_id, rel_id, tail_id, begin, end`
- `alignment_all.tsv` — gold pairs
- `entity2id_*.tsv`, `relation2id.tsv`

### Colab setup
1. Runtime → **GPU**
2. Put `wikidata12k_crosslingual.zip` on Drive
3. Set `LANG_PAIR` (default `en_fr`)
4. Run top to bottom

**Note:** This is a research reconstruction/port; report results with that caveat.

In [13]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

import os
import re
import gc
import zipfile
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable a Colab GPU runtime.")

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [14]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# ============================================================
# 3. Configuration
# ============================================================

LANG_PAIR = "en_ja"  # en_fr | en_fa | en_zh | en_ja

ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/wikidata12k_crosslingual.zip"),
    Path("/content/wikidata12k_crosslingual.zip"),
]
EXTRACT_ROOT = Path("/content/wikidata12k_crosslingual")

# TEA-GNN / RREA-style hyperparameters (paper-like defaults)
TRAIN_SEED = 200
DIM = 100
DEPTH = 2
DROPOUT = 0.30
GAMMA = 3.0
LR = 0.005
EPOCHS_PER_TURN = 1200
TURNS = 5
BOOTSTRAP = True

TOP_K = (1, 5, 10)
CSLS_K = 10

print("LANG_PAIR:", LANG_PAIR)
print("DEVICE:", DEVICE)

LANG_PAIR: en_ja
DEVICE: cuda


In [16]:
# ============================================================
# 4. Extract dataset and locate pair folder
# ============================================================

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    found = list(Path("/content").rglob("wikidata12k_crosslingual.zip"))
    zip_path = found[0] if found else None
if zip_path is None:
    raise FileNotFoundError("wikidata12k_crosslingual.zip not found on Drive or /content")

print("Zip:", zip_path)
if not (EXTRACT_ROOT / LANG_PAIR / "alignment_all.tsv").exists():
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")

PAIR_DIR = EXTRACT_ROOT / LANG_PAIR
if not (PAIR_DIR / "alignment_all.tsv").exists():
    hits = list(Path("/content").rglob(LANG_PAIR))
    hits = [p for p in hits if p.is_dir() and (p / "alignment_all.tsv").exists()]
    if not hits:
        raise FileNotFoundError(LANG_PAIR)
    PAIR_DIR = hits[0]

print("PAIR_DIR:", PAIR_DIR)
for p in sorted(PAIR_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size/1024:.1f} KB")

Zip: /content/drive/MyDrive/wikidata12k_crosslingual.zip
PAIR_DIR: /content/wikidata12k_crosslingual/en_ja
  alignment_all.tsv        561.0 KB
  entity2id_en.tsv         417.0 KB
  entity2id_ja.tsv         359.9 KB
  quadruples_en.tsv        1412.4 KB
  quadruples_ja.tsv        888.4 KB
  relation2id.tsv          0.6 KB


In [17]:
# ============================================================
# 5. Load Wikidata12K pair
# ============================================================

def parse_time_token(tok: str) -> int:
    tok = tok.strip()
    m = re.match(r"^(-?\d+)", tok)
    return int(m.group(1)) if m else 0


def load_quadruples(path):
    triples, entities, relations, times = [], set(), set(), set()
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 5:
                parts = line.split()
            if len(parts) < 5:
                continue
            h, r, t = int(parts[0]), int(parts[1]), int(parts[2])
            ts, te = parse_time_token(parts[3]), parse_time_token(parts[4])
            triples.append((h, r, t, ts, te))
            entities.update((h, t))
            relations.add(r)
            times.update((ts, te))
    return triples, entities, relations, times


def load_alignment(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        next(f)
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                pairs.append((int(parts[0]), int(parts[1])))
    return pairs


lang_a, lang_b = LANG_PAIR.split("_")
triples1, e1_raw, r1_raw, t1_raw = load_quadruples(PAIR_DIR / f"quadruples_{lang_a}.tsv")
triples2, e2_raw, r2_raw, t2_raw = load_quadruples(PAIR_DIR / f"quadruples_{lang_b}.tsv")
all_pairs = load_alignment(PAIR_DIR / "alignment_all.tsv")

print(f"KG1 ({lang_a}): triples={len(triples1)} ents={len(e1_raw)} rels={len(r1_raw)} times={len(t1_raw)}")
print(f"KG2 ({lang_b}): triples={len(triples2)} ents={len(e2_raw)} rels={len(r2_raw)} times={len(t2_raw)}")
print(f"Alignment pairs: {len(all_pairs)}")

KG1 (en): triples=40446 ents=12502 rels=24 times=618
KG2 (ja): triples=25789 ents=8351 rels=23 times=532
Alignment pairs: 8317


In [18]:
# ============================================================
# 6. Remap IDs + build TEA-GNN style matrices
# ============================================================

def make_maps(s):
    ids = sorted(s)
    return {x: i for i, x in enumerate(ids)}


e1_map, e2_map = make_maps(e1_raw), make_maps(e2_raw)
r_map = make_maps(r1_raw | r2_raw)
# time ids: reserve 0 as unknown, shift raw times by +1
all_times = sorted(t1_raw | t2_raw)
time_map = {t: i + 1 for i, t in enumerate(all_times)}  # 1..T
num_times = len(all_times) + 1  # include 0

N1, N2 = len(e1_map), len(e2_map)
NUM_ENTITIES = N1 + N2
NUM_RELS_RAW = len(r_map)
# TEA-GNN doubles relations for reverse edges
NUM_RELS = NUM_RELS_RAW * 2


def remap_triples(triples, e_map, r_map, t_map, entity_offset=0):
    out = []
    for h, r, t, ts, te in triples:
        out.append((
            e_map[h] + entity_offset,
            r_map[r],
            e_map[t] + entity_offset,
            t_map.get(ts, 0),
            t_map.get(te, 0),
        ))
    return out


triples1_m = remap_triples(triples1, e1_map, r_map, time_map, 0)
triples2_m = remap_triples(triples2, e2_map, r_map, time_map, N1)
all_triples = triples1_m + triples2_m

valid_pairs = np.array([
    [e1_map[a], e2_map[b] + N1]
    for a, b in all_pairs if a in e1_map and b in e2_map
], dtype=np.int64)
np.random.shuffle(valid_pairs)

if len(valid_pairs) < TRAIN_SEED:
    raise ValueError(f"valid pairs {len(valid_pairs)} < TRAIN_SEED {TRAIN_SEED}")

train_pair = valid_pairs[:TRAIN_SEED].copy()
dev_pair = valid_pairs[TRAIN_SEED:].copy()

print(f"N1={N1} N2={N2} entities={NUM_ENTITIES}")
print(f"rels_raw={NUM_RELS_RAW} rels_doubled={NUM_RELS} times={num_times}")
print(f"train={len(train_pair)} dev={len(dev_pair)}")

N1=12502 N2=8351 entities=20853
rels_raw=24 rels_doubled=48 times=620
train=200 dev=8117


In [19]:
# ============================================================
# 7. Build adjacency / relation-index / time features (TEA-GNN)
# ============================================================
# Each (h,r,t,ts,te) becomes two directed edges:
#   begin: h -r-> t at time ts
#   end:   t -r_inv-> h at time te   (direction reversed)

ent_size = NUM_ENTITIES
rel_size = NUM_RELS  # doubled

adj_matrix = sp.lil_matrix((ent_size, ent_size))
adj_features = sp.lil_matrix((ent_size, ent_size))
rel_in = np.zeros((ent_size, rel_size))
rel_out = np.zeros((ent_size, rel_size))
time_in = np.zeros((ent_size, num_times))
time_out = np.zeros((ent_size, num_times))

for i in range(ent_size):
    adj_features[i, i] = 1

radj = []  # [h, t, r, tau] directed edges for attention

for h, r, t, ts, te in all_triples:
    # undirected structure for adj
    adj_matrix[h, t] = 1; adj_matrix[t, h] = 1
    adj_features[h, t] = 1; adj_features[t, h] = 1

    # begin edge: h -> t with r, time ts
    radj.append([h, t, r, ts])
    rel_out[h][r] += 1; rel_in[t][r] += 1
    time_out[h][ts] += 1; time_in[t][ts] += 1

    # end edge: t -> h with r+num_raw, time te
    r_inv = r + NUM_RELS_RAW
    radj.append([t, h, r_inv, te])
    rel_out[t][r_inv] += 1; rel_in[h][r_inv] += 1
    time_out[t][te] += 1; time_in[h][te] += 1

# Build r_index / r_val / t_index / t_val grouped by unique (h,t) edge slots
count = -1
s = set()
d = {}
r_index, r_val = [], []
t_index, t_val = [], []
edge_list = []  # unique (h,t) for edge_index

for h, t, r, tau in sorted(radj, key=lambda x: (x[0], x[1])):
    key = f"{h} {t}"
    if key in s:
        r_index.append([count, r]); r_val.append(1)
        t_index.append([count, tau]); t_val.append(1)
        d[count] += 1
    else:
        count += 1
        d[count] = 1
        s.add(key)
        edge_list.append((h, t))
        r_index.append([count, r]); r_val.append(1)
        t_index.append([count, tau]); t_val.append(1)

for i in range(len(r_index)):
    r_val[i] /= d[r_index[i][0]]
    t_val[i] /= d[t_index[i][0]]


def normalize_adj(adj):
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1))
    d_inv = np.power(rowsum, -0.5).flatten()
    d_inv[np.isinf(d_inv)] = 0.0
    d_mat = sp.diags(d_inv)
    return d_mat.dot(adj).transpose().dot(d_mat).T


rel_features = np.concatenate([rel_in, rel_out], axis=1)
time_features = np.concatenate([time_in, time_out], axis=1)
adj_features = normalize_adj(adj_features)
rel_features = normalize_adj(sp.lil_matrix(rel_features))
time_features = normalize_adj(sp.lil_matrix(time_features))

r_index = np.array(r_index)
r_val = np.array(r_val, dtype=np.float32)
t_index = np.array(t_index)
t_val = np.array(t_val, dtype=np.float32)
edge_index_np = np.array(edge_list, dtype=np.int64)

print(f"unique directed edges: {len(edge_list)}")
print(f"r_index rows: {len(r_index)}  t_index rows: {len(t_index)}")
print(f"rel_features shape: {rel_features.shape}  time_features: {time_features.shape}")

unique directed edges: 103930
r_index rows: 132470  t_index rows: 132470
rel_features shape: (20853, 96)  time_features: (20853, 1240)


In [20]:
# ============================================================
# 8. Evaluation helpers
# ============================================================

def csls_sim(Lvec, Rvec, k=10):
    sim_mat = Lvec @ Rvec.T
    if k <= 0:
        return sim_mat
    csls1 = (-np.partition(-sim_mat, k, axis=1)[:, :k]).mean(axis=1)
    csls2 = (-np.partition(-sim_mat.T, k, axis=1)[:, :k]).mean(axis=1)
    return 2 * sim_mat - csls1[:, None] - csls2[None, :]


def get_hits(vec, test_pair, top_k=(1, 5, 10)):
    Lvec = np.array([vec[e1] for e1, e2 in test_pair])
    Rvec = np.array([vec[e2] for e1, e2 in test_pair])
    Lvec = Lvec / np.linalg.norm(Lvec, axis=-1, keepdims=True)
    Rvec = Rvec / np.linalg.norm(Rvec, axis=-1, keepdims=True)
    sim_o = -Lvec.dot(Rvec.T)
    sim = sim_o.argsort(-1)
    top_lr = [0] * len(top_k); MRR_lr = 0.0
    for i in range(Lvec.shape[0]):
        rank = np.where(sim[i] == i)[0][0]
        MRR_lr += 1.0 / (rank + 1)
        for j, k in enumerate(top_k):
            if rank < k:
                top_lr[j] += 1
    print("For each left:")
    for i, k in enumerate(top_k):
        print(f"Hits@{k}: {top_lr[i] / len(test_pair) * 100:.2f}%")
    print(f"MRR: {MRR_lr / Lvec.shape[0]:.3f}")


def csls_get_hits(vec, test_pair, top_k=(1, 5, 10), csls_k=10):
    Lvec = np.array([vec[e1] for e1, e2 in test_pair])
    Rvec = np.array([vec[e2] for e1, e2 in test_pair])
    Lvec = Lvec / np.linalg.norm(Lvec, axis=-1, keepdims=True)
    Rvec = Rvec / np.linalg.norm(Rvec, axis=-1, keepdims=True)
    sim_o = -csls_sim(Lvec, Rvec, k=csls_k)
    sim = sim_o.argsort(-1)
    top_lr = [0] * len(top_k); MRR_lr = 0.0
    for i in range(Lvec.shape[0]):
        rank = np.where(sim[i] == i)[0][0]
        MRR_lr += 1.0 / (rank + 1)
        for j, k in enumerate(top_k):
            if rank < k:
                top_lr[j] += 1
    print("CSLS (left->right):")
    for i, k in enumerate(top_k):
        print(f"Hits@{k}: {top_lr[i] / len(test_pair) * 100:.2f}%")
    print(f"MRR: {MRR_lr / Lvec.shape[0]:.3f}")

print("Eval helpers defined.")

Eval helpers defined.


In [21]:
# ============================================================
# 9. TEA-GNN model (RREA + time-aware attention)
# ============================================================

def sparse_uniform_avg(row_idx, col_idx, emb_table, num_rows):
    out = torch.zeros(num_rows, emb_table.shape[-1], device=emb_table.device)
    counts = torch.zeros(num_rows, device=emb_table.device)
    out.index_add_(0, row_idx, emb_table[col_idx])
    counts.index_add_(0, row_idx, torch.ones_like(row_idx, dtype=torch.float32))
    return out / counts.clamp_min(1.0).unsqueeze(-1)


class TNRGraphAttention(nn.Module):
    """Time-aware + relation-aware reflection attention (TEA-GNN)."""

    def __init__(self, node_size, depth, attn_heads, feat_dim):
        super().__init__()
        self.node_size = node_size
        self.depth = depth
        self.attn_heads = attn_heads
        # kernels for relation-reflection and time-reflection attention
        self.attn_kernels_r = nn.ParameterList()
        self.attn_kernels_t = nn.ParameterList()
        for _ in range(depth):
            for _ in range(attn_heads):
                kr = nn.Parameter(torch.empty(3 * feat_dim, 1))
                kt = nn.Parameter(torch.empty(3 * feat_dim, 1))
                nn.init.xavier_uniform_(kr)
                nn.init.xavier_uniform_(kt)
                self.attn_kernels_r.append(kr)
                self.attn_kernels_t.append(kt)

    @staticmethod
    def _segment_softmax(scores, seg_ids, num_segments):
        seg_max = torch.full((num_segments,), float("-inf"), device=scores.device)
        seg_max.scatter_reduce_(0, seg_ids, scores, reduce="amax", include_self=True)
        seg_max = torch.nan_to_num(seg_max, neginf=0.0)
        exp_scores = torch.exp(scores - seg_max[seg_ids])
        seg_sum = torch.zeros(num_segments, device=scores.device)
        seg_sum.scatter_add_(0, seg_ids, exp_scores)
        return exp_scores / seg_sum[seg_ids].clamp_min(1e-16)

    def forward(self, features, rel_emb, time_emb, edge_index,
                r_index, r_val, t_index, t_val):
        device = features.device
        node_size = self.node_size
        triple_size = edge_index.shape[0]
        src, dst = edge_index[:, 0], edge_index[:, 1]

        outputs = [features]
        feat = F.relu(features)

        for l in range(self.depth):
            head_outs = []
            for h in range(self.attn_heads):
                kr = self.attn_kernels_r[l * self.attn_heads + h]
                kt = self.attn_kernels_t[l * self.attn_heads + h]

                # aggregate relation embeddings per edge slot
                rels_sum = torch.zeros(triple_size, rel_emb.shape[-1], device=device)
                rels_sum.index_add_(0, r_index[:, 0], r_val.unsqueeze(-1) * rel_emb[r_index[:, 1]])
                rels_sum = F.normalize(rels_sum, p=2, dim=1)

                times_sum = torch.zeros(triple_size, time_emb.shape[-1], device=device)
                times_sum.index_add_(0, t_index[:, 0], t_val.unsqueeze(-1) * time_emb[t_index[:, 1]])
                times_sum = F.normalize(times_sum, p=2, dim=1)

                selfs = feat[src]
                neighs = feat[dst]

                # relational reflection (Householder-style, as in RREA)
                bias_r = (neighs * rels_sum).sum(dim=1, keepdim=True) * rels_sum
                neighs_r = neighs - 2 * bias_r

                # temporal reflection
                bias_t = (neighs * times_sum).sum(dim=1, keepdim=True) * times_sum
                neighs_t = neighs - 2 * bias_t

                att_r = (torch.cat([selfs, neighs_r, rels_sum], dim=1) @ kr).squeeze(-1)
                att_t = (torch.cat([selfs, neighs_t, times_sum], dim=1) @ kt).squeeze(-1)
                att = self._segment_softmax(att_r + att_t, src, node_size)

                # combine reflected neighbors
                neighs_comb = 0.5 * (neighs_r + neighs_t)
                msg = neighs_comb * att.unsqueeze(-1)
                new_feat = torch.zeros(node_size, feat.shape[-1], device=device)
                new_feat.index_add_(0, src, msg)
                head_outs.append(new_feat)

            feat = torch.stack(head_outs, dim=0).mean(dim=0)
            feat = F.relu(feat)
            outputs.append(feat)

        return torch.cat(outputs, dim=1)


class TEAGNN(nn.Module):
    def __init__(self, node_size, rel_size, time_size, dim=100, depth=2, dropout=0.3):
        super().__init__()
        self.node_size = node_size
        self.ent_emb = nn.Embedding(node_size, dim)
        self.rel_emb = nn.Embedding(rel_size, dim)
        self.time_emb = nn.Embedding(time_size, dim)
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        nn.init.xavier_uniform_(self.time_emb.weight)

        self.encoder = TNRGraphAttention(node_size, depth, attn_heads=1, feat_dim=dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, edge_index, r_index, r_val, t_index, t_val,
                ent_adj_row, ent_adj_col, rel_adj_row, rel_adj_col,
                time_adj_row, time_adj_col):
        # uniform avg features (official SparseTensor+softmax ≈ mean)
        ent_feat = sparse_uniform_avg(ent_adj_row, ent_adj_col, self.ent_emb.weight, self.node_size)
        rel_feat = sparse_uniform_avg(rel_adj_row, rel_adj_col, self.rel_emb.weight, self.node_size)
        time_feat = sparse_uniform_avg(time_adj_row, time_adj_col, self.time_emb.weight, self.node_size)

        out_e = self.encoder(ent_feat, self.rel_emb.weight, self.time_emb.weight,
                             edge_index, r_index, r_val, t_index, t_val)
        out_r = self.encoder(rel_feat, self.rel_emb.weight, self.time_emb.weight,
                             edge_index, r_index, r_val, t_index, t_val)
        out_t = self.encoder(time_feat, self.rel_emb.weight, self.time_emb.weight,
                             edge_index, r_index, r_val, t_index, t_val)
        out = torch.cat([out_e, out_r, out_t], dim=-1)
        return self.dropout(out)


def align_loss(embeddings, pos_pairs, neg_l, neg_r, gamma):
    l = embeddings[pos_pairs[:, 0]]
    r = embeddings[pos_pairs[:, 1]]
    fl = embeddings[neg_l]
    fr = embeddings[neg_r]

    def l1(a, b):
        return torch.sum(torch.abs(a - b), dim=-1, keepdim=True)

    loss = F.relu(gamma + l1(l, r) - l1(l, fr)) + F.relu(gamma + l1(l, r) - l1(fl, r))
    return loss.sum() / embeddings.shape[0]


def get_train_set(train_pair, node_size, batch_size):
    negative_ratio = batch_size // len(train_pair) + 1
    train_set = np.repeat(train_pair, negative_ratio, axis=0)
    np.random.shuffle(train_set)
    train_set = train_set[:batch_size]
    neg = np.random.randint(0, node_size, train_set.shape)
    return np.concatenate([train_set, neg], axis=-1)


def bootstrap_new_pairs(emb, rest_set_1, rest_set_2):
    Lvec = np.array([emb[e] for e in rest_set_1])
    Rvec = np.array([emb[e] for e in rest_set_2])
    Lvec = Lvec / np.linalg.norm(Lvec, axis=-1, keepdims=True)
    Rvec = Rvec / np.linalg.norm(Rvec, axis=-1, keepdims=True)
    nn_lr = (Lvec @ Rvec.T).argmax(axis=1)
    nn_rl = (Rvec @ Lvec.T).argmax(axis=1)
    new_pairs = []
    for a in range(len(rest_set_1)):
        b = nn_lr[a]
        if nn_rl[b] == a:
            new_pairs.append((rest_set_1[a], rest_set_2[b]))
    return new_pairs

print("TEA-GNN model defined.")

TEA-GNN model defined.


In [22]:
# ============================================================
# 10. Build tensors and model
# ============================================================

edge_index = torch.tensor(edge_index_np, dtype=torch.long, device=DEVICE)
r_index_t = torch.tensor(r_index, dtype=torch.long, device=DEVICE)
r_val_t = torch.tensor(r_val, dtype=torch.float32, device=DEVICE)
t_index_t = torch.tensor(t_index, dtype=torch.long, device=DEVICE)
t_val_t = torch.tensor(t_val, dtype=torch.float32, device=DEVICE)

ent_adj_idx = np.stack(adj_features.nonzero(), axis=1)
rel_adj_idx = np.stack(rel_features.nonzero(), axis=1)
time_adj_idx = np.stack(time_features.nonzero(), axis=1)

ent_adj_row = torch.tensor(ent_adj_idx[:, 0], dtype=torch.long, device=DEVICE)
ent_adj_col = torch.tensor(ent_adj_idx[:, 1], dtype=torch.long, device=DEVICE)
rel_adj_row = torch.tensor(rel_adj_idx[:, 0], dtype=torch.long, device=DEVICE)
rel_adj_col = torch.tensor(rel_adj_idx[:, 1], dtype=torch.long, device=DEVICE)
time_adj_row = torch.tensor(time_adj_idx[:, 0], dtype=torch.long, device=DEVICE)
time_adj_col = torch.tensor(time_adj_idx[:, 1], dtype=torch.long, device=DEVICE)

# rel_emb size: rel_features has 2*rel_size columns (in+out), matching RREA
# TEA-GNN uses doubled relation embeddings for forward+inverse
model = TEAGNN(
    node_size=NUM_ENTITIES,
    rel_size=rel_features.shape[1],  # in+out = 2 * NUM_RELS
    time_size=time_features.shape[1],  # in+out times
    dim=DIM,
    depth=DEPTH,
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.RMSprop(model.parameters(), lr=LR)
batch_size = NUM_ENTITIES

print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

TEAGNN(
  (ent_emb): Embedding(20853, 100)
  (rel_emb): Embedding(96, 100)
  (time_emb): Embedding(1240, 100)
  (encoder): TNRGraphAttention(
    (attn_kernels_r): ParameterList(
        (0): Parameter containing: [torch.float32 of size 300x1 (cuda:0)]
        (1): Parameter containing: [torch.float32 of size 300x1 (cuda:0)]
    )
    (attn_kernels_t): ParameterList(
        (0): Parameter containing: [torch.float32 of size 300x1 (cuda:0)]
        (1): Parameter containing: [torch.float32 of size 300x1 (cuda:0)]
    )
  )
  (dropout): Dropout(p=0.3, inplace=False)
)
Parameters: 2220100


In [23]:
# ============================================================
# 11. Training (multi-turn + optional bootstrap)
# ============================================================

rest_set_1 = [e1 for e1, e2 in dev_pair]
rest_set_2 = [e2 for e1, e2 in dev_pair]
np.random.shuffle(rest_set_1)
np.random.shuffle(rest_set_2)

train_pair_cur = train_pair.copy()

for turn in range(TURNS):
    print(f"\n===== turn {turn} | train pairs={len(train_pair_cur)} =====")
    for i in range(EPOCHS_PER_TURN):
        model.train()
        train_set = get_train_set(train_pair_cur, NUM_ENTITIES, batch_size)
        pos_pairs = train_set[:, :2]
        neg_l, neg_r = train_set[:, 2], train_set[:, 3]

        optimizer.zero_grad()
        emb = model(
            edge_index, r_index_t, r_val_t, t_index_t, t_val_t,
            ent_adj_row, ent_adj_col, rel_adj_row, rel_adj_col,
            time_adj_row, time_adj_col,
        )
        loss = align_loss(
            emb,
            torch.tensor(pos_pairs, dtype=torch.long, device=DEVICE),
            torch.tensor(neg_l, dtype=torch.long, device=DEVICE),
            torch.tensor(neg_r, dtype=torch.long, device=DEVICE),
            GAMMA,
        )
        loss.backward()
        optimizer.step()

        if i % 300 == 299:
            model.eval()
            with torch.no_grad():
                emb_np = model(
                    edge_index, r_index_t, r_val_t, t_index_t, t_val_t,
                    ent_adj_row, ent_adj_col, rel_adj_row, rel_adj_col,
                    time_adj_row, time_adj_col,
                ).cpu().numpy()
            print(f"  [turn {turn} epoch {i+1}] loss={loss.item():.4f}")
            csls_get_hits(emb_np, dev_pair, top_k=TOP_K, csls_k=CSLS_K)

    if BOOTSTRAP and turn < TURNS - 1:
        model.eval()
        with torch.no_grad():
            emb_np = model(
                edge_index, r_index_t, r_val_t, t_index_t, t_val_t,
                ent_adj_row, ent_adj_col, rel_adj_row, rel_adj_col,
                time_adj_row, time_adj_col,
            ).cpu().numpy()
        new_pairs = bootstrap_new_pairs(emb_np, rest_set_1, rest_set_2)
        print(f"new semi-pairs: {len(new_pairs)}")
        if new_pairs:
            train_pair_cur = np.concatenate([train_pair_cur, np.array(new_pairs)], axis=0)
            new_l = {p[0] for p in new_pairs}
            new_r = {p[1] for p in new_pairs}
            rest_set_1 = [e for e in rest_set_1 if e not in new_l]
            rest_set_2 = [e for e in rest_set_2 if e not in new_r]

print("Training complete.")


===== turn 0 | train pairs=200 =====
  [turn 0 epoch 300] loss=0.0019
CSLS (left->right):
Hits@1: 81.73%
Hits@5: 90.69%
Hits@10: 93.41%
MRR: 0.859
  [turn 0 epoch 600] loss=0.0034
CSLS (left->right):
Hits@1: 81.64%
Hits@5: 91.08%
Hits@10: 93.98%
MRR: 0.860
  [turn 0 epoch 900] loss=0.0035
CSLS (left->right):
Hits@1: 81.79%
Hits@5: 90.55%
Hits@10: 93.43%
MRR: 0.859
  [turn 0 epoch 1200] loss=0.0017
CSLS (left->right):
Hits@1: 80.14%
Hits@5: 90.19%
Hits@10: 93.41%
MRR: 0.849
new semi-pairs: 5356

===== turn 1 | train pairs=5556 =====
  [turn 1 epoch 300] loss=0.0020
CSLS (left->right):
Hits@1: 89.40%
Hits@5: 94.87%
Hits@10: 96.55%
MRR: 0.920
  [turn 1 epoch 600] loss=0.0003
CSLS (left->right):
Hits@1: 89.91%
Hits@5: 95.07%
Hits@10: 96.81%
MRR: 0.923
  [turn 1 epoch 900] loss=0.0002
CSLS (left->right):
Hits@1: 90.05%
Hits@5: 95.06%
Hits@10: 96.90%
MRR: 0.924
  [turn 1 epoch 1200] loss=0.0031
CSLS (left->right):
Hits@1: 90.00%
Hits@5: 94.94%
Hits@10: 96.77%
MRR: 0.924
new semi-pairs: 1899

In [24]:
# ============================================================
# 12. Final evaluation + checkpoint
# ============================================================

model.eval()
with torch.no_grad():
    final_emb = model(
        edge_index, r_index_t, r_val_t, t_index_t, t_val_t,
        ent_adj_row, ent_adj_col, rel_adj_row, rel_adj_col,
        time_adj_row, time_adj_col,
    ).cpu().numpy()

print("Embedding shape:", final_emb.shape)
print("\n--- Plain cosine ---")
get_hits(final_emb, dev_pair, top_k=TOP_K)
print("\n--- CSLS ---")
csls_get_hits(final_emb, dev_pair, top_k=TOP_K, csls_k=CSLS_K)

SAVE_PATH = f"/content/drive/MyDrive/teagnn_wikidata12k_{LANG_PAIR}_checkpoint.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "LANG_PAIR": LANG_PAIR,
        "TRAIN_SEED": TRAIN_SEED,
        "DIM": DIM,
        "DEPTH": DEPTH,
        "DROPOUT": DROPOUT,
        "GAMMA": GAMMA,
        "LR": LR,
        "EPOCHS_PER_TURN": EPOCHS_PER_TURN,
        "TURNS": TURNS,
        "BOOTSTRAP": BOOTSTRAP,
    },
}, SAVE_PATH)
print("Saved:", SAVE_PATH)

Embedding shape: (20853, 900)

--- Plain cosine ---
For each left:
Hits@1: 91.56%
Hits@5: 95.64%
Hits@10: 97.04%
MRR: 0.935

--- CSLS ---
CSLS (left->right):
Hits@1: 91.72%
Hits@5: 96.01%
Hits@10: 97.33%
MRR: 0.937
Saved: /content/drive/MyDrive/teagnn_wikidata12k_en_ja_checkpoint.pt


## Notes

- **TEA-GNN** = RREA relational reflection + **time-aware** reflection attention on begin/end quadruples.
- Wikidata12K uses year-like strings (`1926-##-##`) converted to integer years.
- Entity IDs are remapped into disjoint namespaces (KG2 offset by N1).
- Relations are shared across languages then **doubled** for inverse edges.
- Change `LANG_PAIR` and re-run from the config cell for `en_fa` / `en_zh` / `en_ja`.
- Keep the same `TRAIN_SEED` and eval protocol when comparing to RREA / TREA / STEA.